### 직렬화(Serialization)

In [19]:
!pip --version

pip 26.2.1 from D:\hanhwa0902\ex0917\.0917venv\Lib\site-packages\pip (python 3.12)



In [20]:
from dotenv import load_dotenv

load_dotenv()

True

In [21]:
from langchain_teddynote import logging

logging.langsmith("test0917")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0917


In [22]:
import os
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")

In [23]:
print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}") # is_le_serializable: 직력화 가능 여부 확인 매서드

ChatOpenAI: True


In [24]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print(f"ChatOpenAI: {llm.is_lc_serializable()}")

ChatOpenAI: True


In [25]:
chain = prompt | llm

chain.is_lc_serializable()

True

#### 체인 직렬화하기

In [26]:
from langchain_core.load import dumpd, dumps # dumpd: 객체를 딕셔너리로 직렬화 # dumps: 객체를 JSON 문자열로 직렬화

dumpd_chain = dumpd(chain)
dumpd_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-5-mini',
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [27]:
type(dumpd_chain)

dict

In [28]:
dumps_chain = dumps(chain)
dumps_chain

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-5-mini", "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

#### Pickle 파일로 직렬화하고 로드하기

- pickle 파일은 Python 객체를 바이너리 형태로 직렬화하는 포맷

In [29]:
import pickle

with open("fruit_chain.pkl", "wb") as f:
    pickle.dump(dumpd_chain, f)

In [30]:
import json

with open("fruit_chain.json", "w") as fp:
    json.dump(dumpd_chain, fp)

In [31]:
import pickle

with open("fruit_chain.pkl", "rb") as f:
    loaded_chain = pickle.load(f)

In [33]:
from langchain_core.load import load

chain_from_file = load(loaded_chain, allowed_objects="all")

print(chain_from_file.invoke({"fruit": "사과"}))

content='사과의 색상은 품종과 숙성 정도에 따라 다양합니다. 보통 다음과 같습니다.\n\n- 빨강계: 예) 후지, 레드 딜리셔스  \n- 초록계: 예) 그라니 스미스(Granny Smith)  \n- 노랑계: 예) 골든 딜리셔스  \n- 혼합/줄무늬: 빨강과 초록·노랑이 섞이거나 줄무늬가 있는 품종도 많음\n\n참고로 속살은 대체로 흰색 또는 연한 크림색입니다. 특정 품종의 색을 알고 싶으시면 품종명을 알려 주세요.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 410, 'prompt_tokens': 15, 'total_tokens': 425, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOw58WZkXa25HN9gWyqSheUrvqjVH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad2d-a87a-7622-9df5-0917d8508c84-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'inp

In [35]:
from langchain_core.load import load, loads

load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all"
)

load_chain.invoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 다양합니다. 보통 빨간색(예: Red Delicious, Fuji), 초록색(예: Granny Smith), 노란색(예: Golden Delicious)이나 이들 색이 섞인(빨강·노랑, 빨강·초록) 형태가 많습니다. 속은 보통 흰색 또는 크림색입니다. 특정 품종의 색을 알고 싶으시면 품종 이름을 알려주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 298, 'prompt_tokens': 15, 'total_tokens': 313, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOw7UV6Zc52oT1Y8F4aV1G0ibluEI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad2f-e451-7ee0-9ff6-fcfc62caefdd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 

In [37]:
with open("fruit_chain.json", "r") as fp:
    loaded_from_json_chain = json.load(fp)
    loaded_chain = load(loaded_from_json_chain, allowed_objects="all")

In [38]:
load_chain.invoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 품종과 숙성도에 따라 다양합니다. 일반적으로 다음과 같은 색이 많습니다.\n\n- 빨간색(예: 레드 딜리셔스, 후지, 갈라)\n- 초록색(예: 그래니 스미스)\n- 노란색(예: 골든 딜리셔스)\n- 또는 빨강·노랑·초록이 섞인 무늬가 있는 경우도 많음\n\n속(과육)은 보통 흰색에서 크림색을 띱니다. 특정 품종이나 사진을 말씀해 주시면 더 정확히 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 15, 'total_tokens': 418, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOw9ssKLIEmceQ2xuqSV6wAmtNaRG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad32-2764-7581-be54-4e53665746bd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_